# XLKD-01B: Dense/On-the-Fly Mapped Logit-KD Re-entry

XLKD-01 kết thúc hợp lệ với quyết định `stop_sparse_topk_branch`:

- mapped mass đạt gate;
- finite logits và position mask đạt gate;
- top-32/64/128 không giữ đủ mass tại `T=4`;
- không có `top-k` nào được chọn.

Không được hạ ngưỡng sau khi xem kết quả. XLKD-01B là một **adaptive branch mới
được khai báo rõ**, sử dụng lựa chọn được marker STOP cho phép:

> Dense/on-the-fly mapped Logit KD.

Teacher chạy frozen trong từng batch; toàn bộ 9.959 chiều đã ánh xạ được
scatter vào vocabulary 10.080 chiều của S1 và tái chuẩn hóa theo mapped mass.
Không lưu dense-logit corpus, không dùng sparse top-k.

Notebook này chỉ khóa protocol và authorize XLKD-02 fixed-batch
loss/backward preflight. Không load model, không đọc ảnh, không huấn luyện và
không mở Validation/Test.

## Kaggle inputs

1. XLKD-00 output.
2. XLKD-01 output.

Runtime: CPU, Internet Off, không dataset/model/GPU.


## Cell 1 — Discover completed and stopped branch


In [1]:
import csv, hashlib, json, os, time
from datetime import datetime, timezone
from pathlib import Path

STARTED = time.perf_counter()
INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
NOTEBOOK_ROOT = (
    INPUT_ROOT / "notebooks"
    if (INPUT_ROOT / "notebooks").is_dir()
    else INPUT_ROOT
)
OUT = Path(os.environ.get(
    "KLCN_OUTPUT_ROOT",
    "/kaggle/working//xlkd01b_dense_onthefly_reentry",
))
AUDIT = OUT / "audits"
CONFIG = OUT / "configs"
RESULT = OUT / "results"
REPORT = OUT / "reports"
MANIFEST = OUT / "manifests"
for directory in [OUT, AUDIT, CONFIG, RESULT, REPORT, MANIFEST]:
    directory.mkdir(parents=True, exist_ok=True)

def now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

def exactly_one(name):
    matches = sorted(NOTEBOOK_ROOT.rglob(name))
    assert len(matches) == 1, (
        f"Cần đúng một {name} dưới {NOTEBOOK_ROOT}; "
        f"tìm thấy {len(matches)}: {matches}"
    )
    return matches[0].resolve()

def resolve_inside(root, relative_path):
    root = Path(root).resolve()
    target = (root / relative_path).resolve()
    assert target.is_relative_to(root), (root, relative_path, target)
    return target

X00_PATH = exactly_one("XLKD00_EXPANSIONNET_S1_COMPATIBILITY_COMPLETE.json")
X01_PATH = exactly_one("XLKD01_PROBABILITY_MASS_PREFLIGHT_COMPLETE.json")
STOP_PATH = exactly_one("XLKD_SPARSE_TOPK_BRANCH_STOPPED.json")
X00, X01, STOP = map(read_json, [X00_PATH, X01_PATH, STOP_PATH])
assert X00["status"] == "complete"
assert X00["structural_mapping_gate"] == "passed"
assert X01["status"] == "complete"
assert X01["mapped_mass_gate"] is True
assert X01["top_k_gate"] is False
assert X01["overall_gate"] is False
assert X01["selected_top_k"] is None
assert X01["decision"] == "stop_sparse_topk_branch"
assert X01["full_training_authorized"] is False
assert STOP["status"] == "stopped"
assert "Register a new protocol for dense/on-the-fly mapped Logit KD." in (
    STOP["allowed_next_actions"]
)
for marker in [X00, X01, STOP]:
    assert marker["test_set_status"] == "sealed"
    assert marker["test_set_used"] is False

write_json(AUDIT / "xlkd01b_cell01_entry.json", {
    "schema_version": "xlkd01b_cell01_v1",
    "created_at_utc": now(),
    "status": "passed",
    "sparse_branch_status": "stopped",
    "mapped_mass_gate": True,
    "top_k_gate": False,
    "adaptive_reentry_type": "dense_on_the_fly",
    "training_started": False,
    "dataset_read": False,
    "model_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01B CELL 1 — ADAPTIVE RE-ENTRY")
print("=" * 78)
print("Sparse top-k branch       : STOPPED")
print("Mapped mass gate          : PASSED")
print("Top-k gate                : FAILED")
print("New branch                : DENSE / ON-THE-FLY")
print("Training / data / model   : False / False / False")
print("Test Set status           : SEALED")
print("Progress                  : 1 / 5 code cells")


XLKD-01B CELL 1 — ADAPTIVE RE-ENTRY
Sparse top-k branch       : STOPPED
Mapped mass gate          : PASSED
Top-k gate                : FAILED
New branch                : DENSE / ON-THE-FLY
Training / data / model   : False / False / False
Test Set status           : SEALED
Progress                  : 1 / 5 code cells


## Cell 2 — Verify manifests and probability-mass evidence


In [2]:
STARTED = time.perf_counter()

def verify_manifest(marker_path, marker):
    reference = marker["manifest"]["csv"]
    path = resolve_inside(marker_path.parent, reference["relative_path"])
    assert path.is_file() and sha256(path) == reference["sha256"]
    with path.open("r", encoding="utf-8", newline="") as handle:
        rows = list(csv.DictReader(handle))
    assert len(rows) == int(reference["rows"])
    for row in rows:
        artifact = resolve_inside(marker_path.parent, row["relative_path"])
        assert artifact.is_file()
        assert artifact.stat().st_size == int(row["size_bytes"])
        assert sha256(artifact) == row["sha256"]
    return rows

x00_rows = verify_manifest(X00_PATH, X00)
x01_rows = verify_manifest(X01_PATH, X01)
SUMMARY_PATH = (
    X01_PATH.parent / "results" / "xlkd01_probability_mass_summary.json"
).resolve()
assert SUMMARY_PATH.is_file()
SUMMARY = read_json(SUMMARY_PATH)
assert SUMMARY["status"] == "passed"
assert SUMMARY["temperature"] == 4.0
assert SUMMARY["sample_size"] == 1024
assert SUMMARY["valid_positions"] == 10800
assert SUMMARY["gates"]["mapped_mass"] is True
assert SUMMARY["gates"]["sparse_top_k"] is False
assert SUMMARY["gates"]["overall"] is False
assert SUMMARY["finite_logit_rate"] == 1.0
assert SUMMARY["position_mask_correctness"] == 1.0
mapped = SUMMARY["mapped_probability_mass"]
unmapped = SUMMARY["unmapped_probability_mass"]
assert mapped["mean"] >= 0.98 and mapped["p05"] >= 0.95
assert abs(mapped["mean"] + unmapped["mean"] - 1.0) <= 1e-7

write_json(AUDIT / "xlkd01b_cell02_integrity.json", {
    "schema_version": "xlkd01b_cell02_v1",
    "created_at_utc": now(),
    "status": "passed",
    "manifest_rows_verified": len(x00_rows) + len(x01_rows),
    "valid_positions": 10800,
    "mapped_mass_mean": mapped["mean"],
    "mapped_mass_p05": mapped["p05"],
    "unmapped_mass_mean": unmapped["mean"],
    "finite_logit_rate": 1.0,
    "position_mask_correctness": 1.0,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01B CELL 2 — EVIDENCE")
print("=" * 78)
print("Manifest rows verified    :", len(x00_rows) + len(x01_rows))
print("Mapped mass mean / p05    :", mapped["mean"], "/", mapped["p05"])
print("Unmapped mass mean        :", unmapped["mean"])
print("Finite logits / mask      : 100% / 100%")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 2 / 5 code cells")


XLKD-01B CELL 2 — EVIDENCE
Manifest rows verified    : 21
Mapped mass mean / p05    : 0.9967117355377586 / 0.9950725555419921
Unmapped mass mean        : 0.0032882643490490778
Finite logits / mask      : 100% / 100%
Validation / Test         : NOT READ / SEALED
Progress                  : 2 / 5 code cells


## Cell 3 — Quantify sparse failure and dense-storage infeasibility


In [3]:
STARTED = time.perf_counter()
top32 = SUMMARY["top_k_retained_mass"]["32"]
top64 = SUMMARY["top_k_retained_mass"]["64"]
top128 = SUMMARY["top_k_retained_mass"]["128"]
assert top128["mean"] < 0.98 and top128["p05"] < 0.95

# 1,304,640 includes BOS/EOS; decoder prediction positions exclude the first
# token of every one of the 113,287 sequences.
pseudo_token_occurrences = 1304640
pseudo_sequences = 113287
prediction_positions = pseudo_token_occurrences - pseudo_sequences
teacher_vocab = 10000
student_vocab = 10080
fp16_dense_gib = (
    prediction_positions * student_vocab * 2 / (1024 ** 3)
)
fp32_dense_gib = (
    prediction_positions * student_vocab * 4 / (1024 ** 3)
)
assert fp16_dense_gib > 20 and fp32_dense_gib > 40

feasibility = {
    "schema_version": "xlkd01b_dense_feasibility_v1",
    "created_at_utc": now(),
    "sparse_failure": {
        "top32_mean": top32["mean"],
        "top64_mean": top64["mean"],
        "top128_mean": top128["mean"],
        "top128_p05": top128["p05"],
        "threshold_relaxation_allowed": False,
    },
    "dense_offline_storage_estimate": {
        "prediction_positions": prediction_positions,
        "student_vocabulary_size": student_vocab,
        "fp16_gib_before_container_overhead": fp16_dense_gib,
        "fp32_gib_before_container_overhead": fp32_dense_gib,
        "offline_dense_corpus_selected": False,
    },
    "selected_strategy": (
        "Compute frozen Teacher logits on-the-fly for each fixed batch; "
        "scatter mapped mass into Student vocabulary and immediately release "
        "Teacher activations before Student backward."
    ),
    "test_set_status": "sealed",
    "test_set_used": False,
}
FEASIBILITY_PATH = RESULT / "xlkd01b_dense_feasibility.json"
write_json(FEASIBILITY_PATH, feasibility)
write_json(AUDIT / "xlkd01b_cell03_feasibility.json", {
    "schema_version": "xlkd01b_cell03_v1",
    "created_at_utc": now(),
    "status": "passed",
    "top128_mean_retained_mass": top128["mean"],
    "fp16_dense_storage_gib": fp16_dense_gib,
    "fp32_dense_storage_gib": fp32_dense_gib,
    "selected_strategy": "dense_on_the_fly",
    "post_hoc_threshold_relaxation": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01B CELL 3 — FEASIBILITY")
print("=" * 78)
print("Top-128 mean / p05        :", top128["mean"], "/", top128["p05"])
print("Offline dense FP16        :", round(fp16_dense_gib, 2), "GiB")
print("Offline dense FP32        :", round(fp32_dense_gib, 2), "GiB")
print("Selected strategy         : DENSE ON-THE-FLY")
print("Threshold relaxation      : PROHIBITED")
print("Progress                  : 3 / 5 code cells")


XLKD-01B CELL 3 — FEASIBILITY
Top-128 mean / p05        : 0.296975008852228 / 0.11098995544016362
Offline dense FP16        : 22.37 GiB
Offline dense FP32        : 44.74 GiB
Selected strategy         : DENSE ON-THE-FLY
Threshold relaxation      : PROHIBITED
Progress                  : 3 / 5 code cells


## Cell 4 — Lock dense/on-the-fly loss-preflight protocol


In [4]:
STARTED = time.perf_counter()
protocol = {
    "schema_version": "xlkd_dense_onthefly_protocol_v1",
    "created_at_utc": now(),
    "status": "locked",
    "branch_type": "adaptive_after_preregistered_sparse_topk_failure",
    "teacher": "ExpansionNet v2",
    "student": "S1 MobileNetV3-Large",
    "method_name": "Dense On-the-Fly Cross-Vocabulary Mapped Logit KD",
    "scientific_disclosure": (
        "This adaptive branch was registered after observing that top-32/64/128 "
        "failed the XLKD-01 retained-mass gate. No previous threshold was "
        "changed and sparse results remain part of the final report."
    ),
    "teacher_distribution": {
        "temperature": 4.0,
        "native_vocabulary_size": 10000,
        "mapped_teacher_dimensions": 9959,
        "student_vocabulary_size": 10080,
        "mapping_operation": (
            "softmax(teacher_logits/T), index mapped Teacher dimensions, "
            "scatter_add into Student token IDs, divide by mapped mass"
        ),
        "unmapped_mass_policy": (
            "Discard 41 unmapped Teacher dimensions and renormalize per "
            "token position; record mapped mass for every batch."
        ),
        "stop_gradient": True,
        "teacher_frozen": True,
    },
    "loss": {
        "formula": (
            "L = 0.5*CE_human + 0.5*T^2*KL("
            "q_teacher_mapped_renormalized || p_student)"
        ),
        "alpha_human_ce": 0.5,
        "alpha_kd": 0.5,
        "temperature": 4.0,
        "pad_mask_source": "Student human target labels",
        "kl_direction": "teacher_to_student",
    },
    "xlkd02_fixed_batch": {
        "sample_source": "Train+Restval only",
        "fixed_image_count": 4,
        "selection_rule": "SHA256(seed=42 || image_id), first four",
        "physical_batch_size": 1,
        "student_initialization": "S1 seed-42 baseline best checkpoint",
        "required_checks": [
            "teacher and student sequence-position alignment",
            "mapped mass mean >=0.98 and minimum >=0.95 on fixed batch",
            "finite CE, KL*T^2, combined loss and gradients",
            "Teacher has no gradients",
            "weighted KD-to-CE gradient ratio in [0.05, 2.00]",
            "20-step fixed-batch combined-loss decrease >=20%",
            "GPU peak allocated memory <=14 GiB",
        ],
        "success_gate": {
            "mapped_mass_mean_min": 0.98,
            "mapped_mass_minimum_min": 0.95,
            "finite_rate": 1.0,
            "gradient_ratio_min": 0.05,
            "gradient_ratio_max": 2.0,
            "overfit_loss_relative_decrease_min": 0.20,
            "peak_gpu_memory_mib_max": 14336,
        },
    },
    "full_training_authorized": False,
    "offline_logit_corpus_authorized": False,
    "validation_selection_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
PROTOCOL_PATH = CONFIG / "xlkd_dense_onthefly_protocol.json"
write_json(PROTOCOL_PATH, protocol)

alignment = {
    "schema_version": "xlkd01b_thesis_alignment_v1",
    "created_at_utc": now(),
    "status": "aligned_as_adaptive_controlled_ablation",
    "relevance": (
        "The branch tests whether soft response knowledge can be transferred "
        "across incompatible caption vocabularies while retaining the same "
        "Teacher, Student, data split and three-seed evaluation plan."
    ),
    "reporting_requirements": [
        "Report XLKD-01 sparse failure before dense/on-the-fly results.",
        "Label XLKD-01B as adaptive, not preregistered from project start.",
        "Compare against the same S1 baseline and Sequence-KD R1.",
        "Report compute/memory cost of running the frozen Teacher online.",
        "Retain negative results and keep Test sealed.",
    ],
    "test_set_status": "sealed",
    "test_set_used": False,
}
ALIGNMENT_PATH = REPORT / "xlkd01b_thesis_alignment.json"
write_json(ALIGNMENT_PATH, alignment)
write_json(AUDIT / "xlkd01b_cell04_protocol.json", {
    "schema_version": "xlkd01b_cell04_v1",
    "created_at_utc": now(),
    "status": "passed",
    "protocol_locked": True,
    "method": protocol["method_name"],
    "temperature": 4.0,
    "alpha_kd": 0.5,
    "full_training_authorized": False,
    "validation_selection_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "elapsed_seconds": time.perf_counter() - STARTED,
})
print("=" * 78)
print("XLKD-01B CELL 4 — PROTOCOL LOCK")
print("=" * 78)
print("Method                    : DENSE ON-THE-FLY MAPPED LOGIT KD")
print("Temperature / alpha KD    : 4.0 / 0.5")
print("Sparse thresholds changed : False")
print("XLKD-02 scope             : FIXED-BATCH LOSS PREFLIGHT")
print("Full training             : NOT AUTHORIZED")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 4 / 5 code cells")


XLKD-01B CELL 4 — PROTOCOL LOCK
Method                    : DENSE ON-THE-FLY MAPPED LOGIT KD
Temperature / alpha KD    : 4.0 / 0.5
Sparse thresholds changed : False
XLKD-02 scope             : FIXED-BATCH LOSS PREFLIGHT
Full training             : NOT AUTHORIZED
Validation / Test         : NOT READ / SEALED
Progress                  : 4 / 5 code cells


## Cell 5 — Package and authorize XLKD-02 fixed-batch preflight


In [5]:
STARTED = time.perf_counter()
authorization = {
    "schema_version": "xlkd02_dense_mapped_loss_preflight_authorized_v1",
    "created_at_utc": now(),
    "status": "authorized",
    "required_next": (
        "XLKD-02 dense on-the-fly mapped Logit-KD fixed-batch "
        "loss/backward/gradient-ratio/overfit preflight"
    ),
    "protocol": {
        "relative_path": PROTOCOL_PATH.relative_to(OUT).as_posix(),
        "sha256": sha256(PROTOCOL_PATH),
    },
    "source_probability_mass_summary": {
        "source_marker": X01_PATH.name,
        "relative_path_under_source": (
            "results/xlkd01_probability_mass_summary.json"
        ),
        "sha256": sha256(SUMMARY_PATH),
    },
    "student_initialization": "S1 seed-42 baseline best checkpoint",
    "physical_batch_size": 1,
    "fixed_image_count": 4,
    "temperature": 4.0,
    "alpha_kd": 0.5,
    "full_training_authorized": False,
    "validation_selection_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
NEXT_AUTH = OUT / "XLKD02_DENSE_MAPPED_LOGIT_LOSS_PREFLIGHT_AUTHORIZED.json"
write_json(NEXT_AUTH, authorization)

tracked = (
    sorted(AUDIT.glob("*.json"))
    + sorted(CONFIG.glob("*.json"))
    + sorted(RESULT.glob("*"))
    + sorted(REPORT.glob("*"))
)
manifest_rows = []
for path in tracked:
    manifest_rows.append({
        "relative_path": path.relative_to(OUT).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": sha256(path),
    })
manifest_path = MANIFEST / "xlkd01b_manifest.csv"
with manifest_path.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(
        handle, fieldnames=["relative_path", "size_bytes", "sha256"]
    )
    writer.writeheader()
    writer.writerows(manifest_rows)
sums_path = MANIFEST / "SHA256SUMS"
sums_path.write_text(
    "".join(
        f'{row["sha256"]}  {row["relative_path"]}\n'
        for row in manifest_rows
    ),
    encoding="utf-8",
)

marker = {
    "schema_version": "xlkd01b_dense_onthefly_reentry_complete_v1",
    "created_at_utc": now(),
    "status": "complete",
    "sparse_branch_status": "stopped_and_retained",
    "adaptive_branch": "dense_on_the_fly",
    "mapped_mass_gate": True,
    "protocol_locked": True,
    "thesis_alignment": "aligned_as_adaptive_controlled_ablation",
    "xlkd02_authorized": True,
    "xlkd02_authorization_sha256": sha256(NEXT_AUTH),
    "full_training_authorized": False,
    "manifest": {
        "csv": {
            "relative_path": manifest_path.relative_to(OUT).as_posix(),
            "sha256": sha256(manifest_path),
            "rows": len(manifest_rows),
        },
        "sha256sums": {
            "relative_path": sums_path.relative_to(OUT).as_posix(),
            "sha256": sha256(sums_path),
        },
    },
    "training_started": False,
    "dataset_read": False,
    "model_read": False,
    "test_set_status": "sealed",
    "test_set_used": False,
}
MARKER_PATH = OUT / "XLKD01B_DENSE_ONTHEFLY_REENTRY_COMPLETE.json"
write_json(MARKER_PATH, marker)

print("=" * 78)
print("XLKD-01B FINAL STATUS")
print("=" * 78)
print("Status                    : COMPLETE")
print("Sparse result             : FAILED, RETAINED")
print("New adaptive protocol     : DENSE ON-THE-FLY LOCKED")
print("Threshold relaxation      : NONE")
print("XLKD-02 fixed-batch       : AUTHORIZED")
print("Full training             : NOT AUTHORIZED")
print("Validation / Test         : NOT READ / SEALED")
print("Progress                  : 5 / 5 code cells")


XLKD-01B FINAL STATUS
Status                    : COMPLETE
Sparse result             : FAILED, RETAINED
New adaptive protocol     : DENSE ON-THE-FLY LOCKED
Threshold relaxation      : NONE
XLKD-02 fixed-batch       : AUTHORIZED
Full training             : NOT AUTHORIZED
Validation / Test         : NOT READ / SEALED
Progress                  : 5 / 5 code cells
